In [1]:
# Step A: uninstall everything we'll replace
!pip uninstall -q -y torch torchvision torchaudio huggingface_hub transformers peft trl bitsandbytes accelerate datasets wandb tokenizers

# Step B: install PyTorch 2.5.1 built for CUDA 12.4 (matches T4, matches bnb binaries)
!pip install -q --no-cache-dir \
    torch==2.5.1 torchvision==0.20.1 \
    --index-url https://download.pytorch.org/whl/cu124

# Step C: install the ML stack
!pip install -q --no-cache-dir \
    "huggingface_hub==0.27.1" \
    "tokenizers==0.20.3" \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.2" \
    "bitsandbytes==0.45.0" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "wandb==0.18.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 299.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 317.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 324.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 404.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 245.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 319.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 325.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 334.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 332.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 333.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 335.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 326.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from kaggle_secrets import UserSecretsClient
import os
u = UserSecretsClient()
os.environ["HF_TOKEN"] = u.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = u.get_secret("WANDB_API_KEY")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
"""QLoRA fine-tuning of Mistral-7B-Instruct-v0.3 on SentinelOps SFT corpus.
Kaggle-ready: checkpoints every 50 steps, resumes from latest, W&B logged.
Run 2-3 times to complete 3 epochs within the 9-hour Kaggle session cap.
"""
import os, json, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

# ---------- paths (Kaggle layout) ----------
KAGGLE = Path("/kaggle").exists()
if KAGGLE:
    candidates = [
        Path("/kaggle/input/sentinelops-sft"),
        Path("/kaggle/input/datasets/ayushgupta07xx/sentinelops-sft"),
    ]
    DATA_DIR = next((p for p in candidates if (p / "sft_train.jsonl").exists()), candidates[0])
    OUT_DIR  = Path("/kaggle/working/qlora_out")
    print(f"Using DATA_DIR={DATA_DIR}")
else:
    DATA_DIR = Path("data/processed")
    OUT_DIR  = Path("training/llm/qlora_out")

TRAIN_FILE = DATA_DIR / "sft_train.jsonl"
VAL_FILE   = DATA_DIR / "sft_val.jsonl"
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

# ---------- hyperparams ----------
MAX_SEQ_LEN    = 2048
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
EPOCHS         = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 8          # effective batch 16
LR             = 2e-4
WARMUP_RATIO   = 0.03
SAVE_STEPS     = 15
EVAL_STEPS     = 15
LOG_STEPS      = 10
SAVE_TOTAL     = 3          # keep last 3 checkpoints only (disk budget)

# ---------- W&B ----------
os.environ.setdefault("WANDB_PROJECT", "sentinelops")
os.environ.setdefault("WANDB_JOB_TYPE", "qlora_sft")

# ---------- dataset ----------
def load_jsonl(p): return [json.loads(l) for l in open(p)]

def format_example(ex):
    """Mistral-Instruct chat format. Single-turn: [INST] instr + input [/INST] output"""
    user = f"{ex['instruction']}\n\n{ex['input']}"
    return {"text": f"<s>[INST] {user} [/INST] {ex['output']}</s>"}

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)
print(f"Loaded {len(train_raw)} train / {len(val_raw)} val raw pairs")

train_ds = Dataset.from_list(train_raw).map(format_example, remove_columns=list(train_raw[0].keys()))
val_ds   = Dataset.from_list(val_raw).map(format_example,   remove_columns=list(val_raw[0].keys()))
print(f"Formatted. Train sample text[:300]: {train_ds[0]['text'][:300]!r}")

# ---------- tokenizer ----------
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

# ---------- 4-bit quant config (NF4) ----------
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ---------- model ----------
print("Loading base model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model)

# ---------- LoRA config (all linear layers) ----------
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ---------- check for existing checkpoint to resume ----------
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_latest_checkpoint():
    roots = [OUT_DIR]
    if KAGGLE:
        # any attached prior-version output / dataset that contains a qlora_out folder
        for p in Path("/kaggle/input").rglob("qlora_out"):
            roots.append(p)
    all_ckpts = []
    for r in roots:
        all_ckpts.extend(r.glob("checkpoint-*"))
    if not all_ckpts:
        return None
    return sorted(all_ckpts, key=lambda p: int(p.name.split("-")[1]))[-1]

latest = find_latest_checkpoint()
if latest and latest.parent.resolve() != OUT_DIR.resolve():
    import shutil
    dest = OUT_DIR / latest.name
    if not dest.exists():
        print(f"Copying prior checkpoint {latest} -> {dest}")
        shutil.copytree(latest, dest)
    resume_from = str(dest)
elif latest:
    resume_from = str(latest)
else:
    resume_from = None
print(f"Resume from: {resume_from}" if resume_from else "No prior checkpoint; fresh start.")

# ---------- training args ----------
args = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=LOG_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    report_to="wandb",
    run_name="mistral7b-qlora-sentinelops",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
)

print("Starting training...")
trainer.train(resume_from_checkpoint=resume_from)

# ---------- save final adapter ----------
final_dir = OUT_DIR / "final_adapter"
trainer.model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
print(f"Saved final adapter to {final_dir}")

# ---------- auto-push to HF Hub on successful completion ----------
if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ["HF_TOKEN"])
        repo_id = "ayushgupta7777/sentinelops-mistral7b-qlora-adapter"
        api.create_repo(repo_id, exist_ok=True, private=False, repo_type="model")
        api.upload_folder(folder_path=str(final_dir), repo_id=repo_id, repo_type="model")
        print(f"✅ Pushed adapter to https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"HF push failed (non-fatal, checkpoint still in /kaggle/working): {e}")

2026-04-19 02:26:39.368863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776565599.740276      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776565599.839392      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776565600.780103      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776565600.780141      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776565600.780143      25 computation_placer.cc:177] computation placer alr

Using DATA_DIR=/kaggle/input/datasets/ayushgupta07xx/sentinelops-sft
Loaded 252 train / 28 val raw pairs


Map:   0%|          | 0/252 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Formatted. Train sample text[:300]: '<s>[INST] Write a public-facing incident postmortem blog post based on the title and summary below. Include what happened, when it happened, who was affected, the technical root cause, how the issue was mitigated, and what steps are being taken to prevent recurrence.\n\nTitle: Post Mortem: The Ugly, t'


...

Saved final adapter to /kaggle/working/qlora_out/final_adapter


Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

✅ Pushed adapter to https://huggingface.co/ayushgupta7777/sentinelops-mistral7b-qlora-adapter


In [4]:
import huggingface_hub, transformers, bitsandbytes
print("hf_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer
print("imports OK")

hf_hub: 0.27.1
transformers: 4.46.3
bitsandbytes: 0.45.0
imports OK


In [5]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch: 2.5.1+cu124
cuda: 12.4
cuda available: True
device: Tesla T4
